# Landsat VHI — Turkish Thrace · v4 (district-level, CORINE-211 masked)
### The fix for the yield calibration

**Why v4.** In v3, VHI was sampled in a 1.5 km buffer around each station with **no land-cover
mask**. Diagnostics on the real output showed the buffers averaged mixed land cover (NDVI peaked
only ~0.4 with amplitude 0.13–0.22, vs ~0.7 / amplitude ~0.45 for real rainfed wheat), which
diluted the drought signal — so VHI–yield correlation was ~0 and α calibration was degenerate.

v4 matches the manuscript's method and the yield's spatial unit:
1. **District polygons** (GAUL admin-2) matched to the yield districts — the same spatial unit as
   the yield data (Edirne, Uzunköprü, İpsala, Lüleburgaz, Kırklareli, Tekirdağ, Çorlu, Sarıyer).
2. **CORINE 211** (non-irrigated arable land) mask, so only rainfed cropland pixels contribute.
3. District-level zonal-mean NDVI/LST → VCI/TCI/VHI that actually track the district's wheat.
4. α calibrated per district against **detrended** yield.

> **CORINE coverage:** §5 checks whether CORINE 211 exists over Thrace. If it returns ~0 pixels
> (the GEE Copernicus asset does not always include Turkey), switch to the ESA WorldCover fallback
> in that cell before extracting.


## 0 · Setup

In [ ]:
!pip install -q geemap earthengine-api xlrd openpyxl
import ee, geemap, numpy as np, pandas as pd, unicodedata
import matplotlib.pyplot as plt
from scipy import sparse, stats
from scipy.sparse.linalg import splu
try:
    import statsmodels.formula.api as smf; HAVE_SM=True
except Exception: HAVE_SM=False

ee.Authenticate()
ee.Initialize(project='YOUR_GEE_PROJECT')
print('EE:', ee.String('OK').getInfo(), '| statsmodels:', HAVE_SM)

## 1 · ROI (Thrace outline — for `filterBounds` and CORINE coverage check)

In [ ]:
roi = ee.Geometry.MultiPolygon([[[[26.05230399900006, 39.85002500000002], [26.07685120000008, 39.83561160000011], [26.083213900000032, 39.80120859900012], [26.061342500000137, 39.78746060000009], [25.96385900000007, 39.84030499900007], [26.05230399900006, 39.85002500000002]]], [[[26.05978389900008, 39.939909700000044], [26.074661900000137, 39.93865710000017], [26.05507370000015, 39.936683699000014], [26.05978389900008, 39.939909700000044]]], [[[25.949912000000154, 40.24571599900008], [25.95346899900005, 40.23019799900004], [25.97922200000005, 40.222683000000075], [25.971517000000063, 40.153080000000045], [26.01783300000011, 40.159251999000105], [25.99370919900008, 40.13234209900003], [25.73880030000015, 40.091379799000094], [25.668508999000096, 40.12995499900006], [25.679143000000067, 40.16004899900014], [25.78719999900005, 40.21671699900003], [25.845881000000134, 40.216700999000125], [25.949912000000154, 40.24571599900008]]], [[[28.771527899000034, 40.98096789900018], [28.77234320000008, 40.98129119900011], [28.772620999000026, 40.97902679900017], [28.774618199000088, 40.97762610000018], [28.76710060000005, 40.98208039900004], [28.771527899000034, 40.98096789900018]]], [[[27.199738300000092, 42.080394900000044], [27.293635500000107, 42.09738190000007], [27.33590250000003, 42.0602128000001], [27.36483860000004, 42.057856300000026], [27.394520300000067, 42.00487419900003], [27.468489300000044, 41.9587146990001], [27.51061180000005, 41.95866270000016], [27.560672600000032, 41.904734199000075], [27.572697900000037, 41.90529209900001], [27.561550100000147, 41.914986800000065], [27.57676299900004, 41.920024000000126], [27.565947900000026, 41.92379580000005], [27.574344300000007, 41.93722279899998], [27.613991800000065, 41.93888529900005], [27.624472600000047, 41.958909299000084], [27.672453299000097, 41.956453199000066], [27.70409710000007, 41.9705738990001], [27.69727279900019, 41.97841329900018], [27.737887300000068, 41.98004480000009], [27.792442799000185, 41.94793759900017], [27.83531220000009, 41.9476725990001], [27.81798259900006, 41.98671939900015], [27.852507699000114, 42.0019871990001], [27.855222700000127, 41.99103489900017], [27.860000200000172, 42.0063715], [27.912147799000024, 41.97364740000012], [27.97627189900004, 41.990083200000015], [28.029384099000026, 41.983212799000114], [28.057436999000174, 41.88459329900019], [27.996510799000077, 41.88623179900003], [27.981840799000054, 41.862223700000186], [27.98687190000004, 41.822635100000184], [28.046362800000054, 41.72976440000019], [28.10214040000011, 41.675213499999984], [28.095398900000077, 41.64121620000009], [28.14508390000003, 41.58379090000011], [28.182792600000084, 41.57105299900019], [28.220717299000114, 41.524764299000026], [28.27240150000017, 41.496445000000165], [28.62587570000011, 41.35472219900015], [28.68312470000012, 41.34850450000005], [28.967595500000073, 41.25388569900008], [29.078123500000117, 41.25397809900011], [29.11134849900003, 41.23839630000003], [29.105706300000065, 41.20670070000017], [29.03601840000016, 41.155944199000146], [29.071641800000123, 41.12454779900014], [29.05427400000019, 41.11382779900009], [29.056448399000033, 41.082567899000026], [29.03268430000014, 41.04982379900002], [28.97660050000013, 41.02107450000011], [28.940546100000176, 41.04666610000015], [28.948350499000128, 41.05724109900018], [28.943823398999996, 41.064927699000066], [28.939581800000155, 41.042634099000054], [28.985559499000033, 41.00616070000018], [28.936718000000155, 41.00122829899999], [28.875225099000147, 40.968910199000106], [28.847301500000185, 40.97351460000016], [28.82983780000012, 40.9546927990001], [28.77219390000016, 40.98793979900006], [28.739935200000048, 40.97141139900015], [28.62135469900005, 40.960634099000174], [28.59550620000016, 40.9734304000001], [28.58844910000016, 41.01682190000014], [28.56327880000015, 41.01665560000009], [28.541237099000114, 40.98458159900002], [28.416296000000045, 41.04262329900007], [28.228129100000103, 41.0780090990001], [28.024463400000116, 41.039254600000106], [27.99108069900018, 41.01879829899997], [27.956491000000028, 40.96422500000017], [27.877902000000063, 40.96602999900006], [27.838403000000028, 40.992821999000114], [27.741788300000053, 41.01355510000002], [27.512444600000038, 40.9747268000001], [27.469678999000166, 40.91344340000012], [27.454593999999986, 40.84735329900019], [27.37885789900014, 40.79716449900013], [27.28707519900007, 40.686878600000114], [27.146797500000105, 40.615542399000105], [27.02878920000012, 40.59335940000011], [26.995415999999977, 40.55617879900018], [26.902581799000075, 40.542279899000164], [26.737173299000062, 40.47763570000018], [26.69292550000017, 40.44966859900012], [26.697617599000182, 40.42054140000005], [26.628241899000102, 40.390198199000054], [26.6315125000001, 40.36198329900003], [26.588343300000133, 40.32053070000012], [26.512383499000066, 40.28944530000007], [26.423585299000024, 40.22105859900006], [26.35842370000006, 40.20337629900018], [26.38107250000013, 40.14385519899997], [26.29589320000008, 40.0849980000001], [26.174836900000173, 40.04325570000009], [26.16616720000013, 40.05355349900009], [26.278993799999967, 40.21939199899998], [26.27737690000015, 40.255623399], [26.229187400000114, 40.28885070000007], [26.24596350000013, 40.30895529900005], [26.21538269900003, 40.316299400000105], [26.59482390000005, 40.50383140000014], [26.705662300000085, 40.51577939900005], [26.746857599000066, 40.562020699000186], [26.77452949999997, 40.54994430000005], [26.818161199000087, 40.57184289900016], [26.845964599000183, 40.61131139900016], [26.78322060000005, 40.666042400000094], [26.701992200000177, 40.63802529900016], [26.606857599000023, 40.64008339999998], [26.532887900000105, 40.59218820000018], [26.450678899000025, 40.61990850000012], [26.418006400000024, 40.602731299000084], [26.382260400000007, 40.61202900000012], [26.292535000000044, 40.591297900000086], [26.218694500000026, 40.60108950000006], [26.140751100000045, 40.59071140000003], [26.0795222000001, 40.612611599000104], [26.06126520000015, 40.63812600000006], [26.065149399000063, 40.673072799000124], [26.059369000000117, 40.68815089900011], [26.049998499000083, 40.69529809900013], [26.048670100000038, 40.698949500000026], [26.05481129900005, 40.695686900000055], [26.067661900000076, 40.70567529900006], [26.05444499900011, 40.69899100000009], [26.035236800000177, 40.729536099000086], [26.123721199999977, 40.742598999000165], [26.15007770000011, 40.802168000000165], [26.219634300000166, 40.83424109900011], [26.20342049900006, 40.850508599000136], [26.22340370000012, 40.86678200000006], [26.222806800000114, 40.88991360000011], [26.25021989900017, 40.881410899000116], [26.26244639900017, 40.914750900000115], [26.29081419900001, 40.89334770000005], [26.302189500000054, 40.90402889900014], [26.288074200000096, 40.93053909900016], [26.36010130000011, 40.9465143000001], [26.325996600000053, 40.98393120000014], [26.349401799000077, 40.99209419900012], [26.36434689900011, 41.02134080000013], [26.32653919900008, 41.04244850000009], [26.311041600000067, 41.07691520000009], [26.33595940000015, 41.10696429900008], [26.303982799000153, 41.11345400000016], [26.33605019999999, 41.14382789900003], [26.30750680000017, 41.1742344000001], [26.324646200000018, 41.18697660000015], [26.324378900000113, 41.25384879900008], [26.402088000000106, 41.25296450000019], [26.519810300000188, 41.340116300000034], [26.587949800000104, 41.32264400000014], [26.62925680000012, 41.35150049900005], [26.629970699000125, 41.41607779900011], [26.61308310000004, 41.424820700000055], [26.591486500000144, 41.52619770000007], [26.591661900000133, 41.61028429900006], [26.530917198999987, 41.62187850000004], [26.48818640000019, 41.6545225000001], [26.489650000000154, 41.672773300000074], [26.358855300000073, 41.70933549900019], [26.329021000000182, 41.73630570000017], [26.339916500000186, 41.77566940000003], [26.377427100000148, 41.82212679900016], [26.538798000000043, 41.82349019900016], [26.58054959900005, 41.87182959900014], [26.567132300000026, 41.885973500000034], [26.584292500000174, 41.90359039900011], [26.562395199000036, 41.925979200000086], [26.623481700000127, 41.97989720000015], [26.751303000000064, 41.958484699999985], [26.792605599000183, 41.990113899000164], [26.846865400000183, 41.969793500000094], [26.87132379900015, 41.99079369900011], [26.965752899000165, 41.99705629900012], [26.98173860000003, 42.037481899000056], [27.025690300000065, 42.056052299000044], [27.024665400000174, 42.08134439900016], [27.066748000000075, 42.09319910000005], [27.20275260000011, 42.059582800000044], [27.199738300000092, 42.080394900000044]]]])
print('ROI area (km^2):', round(roi.area(maxError=100).getInfo()/1e6,1))

## 2 · Parameters

In [ ]:
START_YEAR = 1985
END_YEAR   = 2024
MONTHS     = list(range(1,13))
SENSORS    = ['L5','L7','L8','L9']
HARMONIZE  = True
L7_INCLUDE_SLC_OFF = True
CLOUD_BITS = True
SCALE_M    = 30

HANDOVER_SPLIT = '2013-01-01'
DELTA_LST_POST = 0.0

CROP       = 'Wheat'          # 'Wheat' or 'Sunflower' (must match a sheet in Yield.xls)
GS_MONTHS  = [4,5,6]          # wheat drought window (spring); sunflower ~[6,7,8]
ALPHA_GRID = np.linspace(0,1,101)
MIN_YEARS  = 6
R_MIN      = 0.20
DETREND_YIELD = True

WH_LAMBDA  = 1.5
WH_ORDER   = 2

# Yield districts -> Turkish province (for GAUL filtering). Folded names matched in §4.
DISTRICT_PROVINCE = {
    'Edirne':'Edirne','Uzunkopru':'Edirne','Ipsala':'Edirne',
    'Luleburgaz':'Kirklareli','Kirklareli':'Kirklareli',
    'Tekirdag':'Tekirdag','Corlu':'Tekirdag','Sariyer':'Istanbul'}

## 3 · Landsat C2 L2 collection (lazy)

In [ ]:
SR_SCALE,SR_OFF=0.0000275,-0.2; ST_SCALE,ST_OFF=0.00341802,149.0
SENSOR_DATES={'L5':('1984-03-01','2012-05-05'),'L7':('1999-05-28','2022-04-06'),
              'L8':('2013-03-18','2025-12-31'),'L9':('2021-10-31','2025-12-31')}
L7_SLC_OFF='2003-05-31'
CFG={'L5':('LANDSAT/LT05/C02/T1_L2',['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7','ST_B6','QA_PIXEL']),
     'L7':('LANDSAT/LE07/C02/T1_L2',['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7','ST_B6','QA_PIXEL']),
     'L8':('LANDSAT/LC08/C02/T1_L2',['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','ST_B10','QA_PIXEL']),
     'L9':('LANDSAT/LC09/C02/T1_L2',['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','ST_B10','QA_PIXEL'])}
COMMON=['blue','green','red','nir','swir1','swir2','thermal','QA_PIXEL']; OPT=['blue','green','red','nir','swir1','swir2']
ROY_S={'blue':0.8474,'green':0.8483,'red':0.9047,'nir':0.8462,'swir1':0.8937,'swir2':0.9071}
ROY_I={'blue':0.0003,'green':0.0088,'red':0.0061,'nir':0.0412,'swir1':0.0254,'swir2':0.0172}

def mask_qa(img):
    qa=img.select('QA_PIXEL')
    bad=(qa.bitwiseAnd(1<<1).neq(0).Or(qa.bitwiseAnd(1<<2).neq(0)).Or(qa.bitwiseAnd(1<<3).neq(0))
         .Or(qa.bitwiseAnd(1<<4).neq(0)).Or(qa.bitwiseAnd(1<<5).neq(0)))
    return img.updateMask(bad.Not())

def prep(sensor):
    coll_id,bands=CFG[sensor]; s0,s1=SENSOR_DATES[sensor]
    lo=f'{max(int(s0[:4]),START_YEAR)}-01-01'; hi=f'{min(int(s1[:4]),END_YEAR)}-12-31'
    coll=ee.ImageCollection(coll_id).filterBounds(roi).filterDate(lo,hi)
    if sensor=='L7' and not L7_INCLUDE_SLC_OFF: coll=coll.filterDate(s0,L7_SLC_OFF)
    if CLOUD_BITS: coll=coll.map(mask_qa)
    def _p(img):
        img=img.select(bands,COMMON); opt=img.select(OPT).multiply(SR_SCALE).add(SR_OFF)
        if HARMONIZE and sensor in ('L5','L7'):
            opt=opt.multiply(ee.Image([ROY_S[b] for b in OPT])).add(ee.Image([ROY_I[b] for b in OPT]))
        lst=img.select('thermal').multiply(ST_SCALE).add(ST_OFF).subtract(273.15).rename('LST')
        ndvi=opt.normalizedDifference(['nir','red']).rename('NDVI')
        return ndvi.addBands(lst).copyProperties(img,['system:time_start'])
    return coll.map(_p)

landsat=prep(SENSORS[0])
for s in SENSORS[1:]: landsat=landsat.merge(prep(s))
landsat=landsat.select(['NDVI','LST'])
print('Collection built (lazy).')

## 4 · District polygons from GAUL, matched to yield districts

Pulls Turkish admin-2 names once (small), matches them to the yield districts by ASCII-folded
name within the right province, and builds a district FeatureCollection tagged with the yield key.
**Check the printed match table** — if any district is unmatched or wrong (e.g. Tekirdağ merkez may
be 'Süleymanpaşa' in GAUL, or Sarıyer resolves oddly), we switch to uploaded district shapefiles.

In [ ]:
def fold(s):
    return ''.join(c for c in unicodedata.normalize('NFKD',str(s)) if not unicodedata.combining(c)).upper().strip()

gaul = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM0_NAME','Turkey'))
# pull just the names we might need (filter to the 4 provinces client-side via a broad grab)
prov_folded = {fold(p) for p in DISTRICT_PROVINCE.values()}
names_df = geemap.ee_to_df(gaul, columns=['ADM1_NAME','ADM2_NAME'])
names_df['adm1_f']=names_df['ADM1_NAME'].map(fold); names_df['adm2_f']=names_df['ADM2_NAME'].map(fold)

matches=[]
for dist,prov in DISTRICT_PROVINCE.items():
    cand = names_df[names_df['adm1_f']==fold(prov)]
    hit = cand[cand['adm2_f']==fold(dist)]
    if len(hit)==0:  # loosen: contains
        hit = cand[cand['adm2_f'].str.contains(fold(dist)[:6], na=False)]
    if len(hit)==0 and dist=='Tekirdag':  # merkez renamed
        hit = cand[cand['adm2_f'].str.contains('SULEYMAN', na=False)]
    gaul_adm2 = hit['ADM2_NAME'].iloc[0] if len(hit) else None
    matches.append({'yield_district':dist,'province':prov,'gaul_ADM2':gaul_adm2,'matched':gaul_adm2 is not None})
match_tbl=pd.DataFrame(matches)
print(match_tbl.to_string(index=False))
unmatched=match_tbl[~match_tbl.matched]['yield_district'].tolist()
print('\nUnmatched:', unmatched if unmatched else 'none — all districts found')

# Build district FC tagged with the yield key
pairs = [(r.yield_district, r.gaul_ADM2) for r in match_tbl.itertuples() if r.matched]
def make_district(key, adm2):
    f = gaul.filter(ee.Filter.eq('ADM2_NAME', adm2)).first()
    return ee.Feature(f.geometry(), {'name': key})
districts = ee.FeatureCollection([make_district(k,a) for k,a in pairs])
print('District polygons built:', districts.size().getInfo())

## 5 · CORINE 211 rainfed-cropland mask  ⟵ *check coverage before extracting*

CORINE class 211 = non-irrigated arable land. The check counts 211 pixels inside the ROI. If it is
~0, the GEE Copernicus CORINE asset does not cover Turkey here — **switch to the ESA WorldCover
fallback** (uncomment) before running §7.

In [ ]:
# --- Primary: CORINE 211 (matches manuscript) ---
corine = ee.Image('COPERNICUS/CORINE/V20/100m/2018').select('landcover')
cropland = corine.eq(211).selfMask().rename('cropland')

# Coverage check
cnt = cropland.reduceRegion(reducer=ee.Reducer.count(), geometry=roi, scale=100, maxPixels=1e10).getInfo()
n211 = list(cnt.values())[0] if cnt else 0
print('CORINE 211 pixels in ROI:', n211)
if not n211:
    print('*** CORINE has no data over Thrace — use the ESA WorldCover fallback below. ***')
else:
    print('CORINE 211 coverage OK — proceed.')

# --- Fallback: ESA WorldCover cropland (class 40), 10 m, global (uncomment if CORINE is empty) ---
# wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
# cropland = wc.eq(40).selfMask().rename('cropland')
# cnt = cropland.reduceRegion(ee.Reducer.count(), roi, 10, maxPixels=1e10).getInfo()
# print('ESA WorldCover cropland pixels in ROI:', list(cnt.values())[0])

## 6 · (numbering parity)

In [ ]:
# extraction is §7

## 7 · Chunked district-level extraction (CORINE-masked)

For each year, mask NDVI/LST to cropland and take the **zonal mean over each district polygon**,
one year at a time (memory-safe). One value per (district, scene).

In [ ]:
def sample_year(year):
    coll = landsat.filterDate(f'{year}-01-01', f'{year}-12-31')
    def samp(img):
        masked = img.updateMask(cropland)
        t = img.get('system:time_start')
        return masked.reduceRegions(collection=districts, reducer=ee.Reducer.mean(), scale=SCALE_M) \
                     .map(lambda f: f.set('millis', t))
    try:
        return geemap.ee_to_df(coll.map(samp).flatten())
    except Exception as e:
        print(f'  {year}: skipped ({str(e)[:60]})'); return pd.DataFrame()

frames=[]
for y in range(START_YEAR, END_YEAR+1):
    d=sample_year(y); frames.append(d)
    print(f'{y}: {len(d):>3} district-scene rows')
scenes=pd.concat(frames, ignore_index=True).dropna(subset=['millis'])
scenes['date']=pd.to_datetime(scenes['millis'],unit='ms')
scenes['year']=scenes.date.dt.year; scenes['month']=scenes.date.dt.month
print('\nTotal rows:', len(scenes))
scenes[['name','date','NDVI','LST']].head()

## 8 · Monthly composites + NDVI sanity check

After CORINE masking, the NDVI seasonal amplitude should now look like rainfed wheat
(spring peak ~0.6–0.8, summer crash). Compare to the v3 diluted values (~0.4, amp 0.13–0.22).

In [ ]:
monthly=(scenes.groupby(['name','year','month'])
         .agg(NDVI=('NDVI','max'),LST=('LST','mean'),n_scenes=('NDVI','size')).reset_index())
monthly['date']=pd.to_datetime(dict(year=monthly.year,month=monthly.month,day=1))

clim=monthly.groupby(['name','month'])['NDVI'].mean().unstack()
print('Monthly mean NDVI by district (expect spring peak ~0.6-0.8 on cropland):')
print((clim[[m for m in range(1,13)]].round(2)).to_string())
print('\nAmplitude (max-min):')
print(((clim.max(axis=1)-clim.min(axis=1)).round(2)).to_string())

## 9 · Handover diagnostic (robust) + LST correction

In [ ]:
def handover_report(df, split=HANDOVER_SPLIT):
    d=df.dropna(subset=['LST']).copy(); sy=pd.Timestamp(split).year
    d['post']=(d['year']>=sy).astype(int); d['yr']=d['year']-d['year'].mean(); d['mo']=d['month'].astype('category')
    rows=[]
    for nm,g in d.groupby('name'):
        gg=g.copy(); gg['mo']=gg['month'].astype('category')
        if HAVE_SM and gg['post'].nunique()==2:
            r=smf.ols('LST ~ C(mo) + yr + post', data=gg).fit()
            rows.append({'district':nm,'step_C':round(r.params['post'],2),'p':round(r.pvalues['post'],4),
                         'sig':r.pvalues['post']<0.05})
        else:
            rows.append({'district':nm,'step_C':np.nan,'p':np.nan,'sig':False})
    return pd.DataFrame(rows)

hand=handover_report(monthly)
print(hand.to_string(index=False))
print('\nApply DELTA_LST_POST (§2) only if a consistent step appears across districts.')

monthly_c=monthly.copy()
post=monthly_c['date']>=pd.Timestamp(HANDOVER_SPLIT)
monthly_c.loc[post,'LST']=monthly_c.loc[post,'LST']-DELTA_LST_POST

## 10 · Climatology → VCI, TCI

In [ ]:
cl=(monthly_c.groupby(['name','month'])
    .agg(NDVI_min=('NDVI','min'),NDVI_max=('NDVI','max'),LST_min=('LST','min'),LST_max=('LST','max')).reset_index())
m=monthly_c.merge(cl,on=['name','month'],how='left')
dv=(m.NDVI_max-m.NDVI_min).replace(0,np.nan); dt=(m.LST_max-m.LST_min).replace(0,np.nan)
m['VCI']=(100*(m.NDVI-m.NDVI_min)/dv).clip(0,100)
m['TCI']=(100*(m.LST_max-m.LST)/dt).clip(0,100)
print('VCI/TCI computed. VCI [%.0f,%.0f] TCI [%.0f,%.0f]'%(m.VCI.min(),m.VCI.max(),m.TCI.min(),m.TCI.max()))

## 11 · Load yield (Excel) + per-district α calibration (detrended)

Upload **Yield.xls** to the Colab session. Yield is district-level, so it aligns directly with the
district VHI. Yield is detrended before calibration (removes the 2004→2024 technology trend).

In [ ]:
yx=pd.read_excel('Yield.xls', sheet_name=CROP, header=0)
yield_df=yx.melt(id_vars='Year', var_name='name', value_name='yield').rename(columns={'Year':'year'})
yield_df=yield_df[yield_df.name.isin(m.name.unique())]
print(f'{CROP} yield: {yield_df.name.nunique()} districts, {int(yield_df.year.min())}-{int(yield_df.year.max())}')

def detrend(sr):
    x=sr.index.values.astype(float); y=sr.values.astype(float); ok=np.isfinite(y)
    if ok.sum()<3: return sr
    c=np.polyfit(x[ok],y[ok],1); return pd.Series(y-np.polyval(c,x),index=sr.index)
def gs_annual(df,nm,col):
    g=df[(df.name==nm)&(df.month.isin(GS_MONTHS))]; return g.groupby('year')[col].mean()
def calibrate_alpha(vci,tci,ref,grid=ALPHA_GRID,min_n=MIN_YEARS):
    mm=np.isfinite(vci)&np.isfinite(tci)&np.isfinite(ref)
    if mm.sum()<min_n: return np.nan,np.nan,int(mm.sum())
    a,b,yv=vci[mm],tci[mm],ref[mm]; ba,br=np.nan,-np.inf
    for al in grid:
        v=al*a+(1-al)*b
        if np.std(v)==0: continue
        r=np.corrcoef(v,yv)[0,1]
        if r>br: br,ba=r,al
    return ba,br,int(mm.sum())

cal=[]
for nm in sorted(m.name.unique()):
    vci_a=gs_annual(m,nm,'VCI'); tci_a=gs_annual(m,nm,'TCI')
    ya=yield_df[yield_df.name==nm].set_index('year')['yield']
    if DETREND_YIELD: ya=detrend(ya)
    idx=vci_a.index.intersection(tci_a.index).intersection(ya.index)
    a,r,n=calibrate_alpha(vci_a.reindex(idx).values,tci_a.reindex(idx).values,ya.reindex(idx).values)
    rv=np.corrcoef(vci_a.reindex(idx).values,ya.reindex(idx).values)[0,1] if len(idx)>3 else np.nan
    cal.append({'district':nm,'alpha':round(a,3) if np.isfinite(a) else np.nan,
                'r_VHI':round(r,3) if np.isfinite(r) else np.nan,'r_VCI':round(rv,2),'n':n})
alpha_tbl=pd.DataFrame(cal)
alpha_tbl['trusted']=(alpha_tbl.r_VHI.abs()>=R_MIN)&alpha_tbl.alpha.notna()
alpha_mean=np.nanmean(alpha_tbl.loc[alpha_tbl.trusted,'alpha']) if alpha_tbl.trusted.any() else np.nanmean(alpha_tbl.alpha)
print(alpha_tbl.to_string(index=False))
print(f'\n{int(alpha_tbl.trusted.sum())}/{len(alpha_tbl)} districts pass |r|>={R_MIN}. Mean α = {alpha_mean:.3f}')
print('If r_VHI is now moderate/high, the CORINE mask restored the drought signal (v3 was ~0).')

## 12 · VHI with calibrated α + Whittaker smoothing

In [ ]:
def whittaker_smooth(y,w,lmbda,d=2):
    y=np.asarray(y,float); w=np.asarray(w,float); n=len(y)
    E=sparse.eye(n,format='csc'); D=E.copy()
    for _ in range(d): D=D[1:]-D[:-1]
    W=sparse.diags(w,0,format='csc'); A=(W+lmbda*(D.T@D)).tocsc()
    return splu(A).solve(w*np.where(w>0,y,0.0))
def smooth_series(g,col):
    g=g.sort_values('date'); full=pd.date_range(g.date.min(),g.date.max(),freq='MS')
    s=g.set_index('date').reindex(full); y=s[col].values; w=np.isfinite(y).astype(float)
    return pd.DataFrame({'date':full,col:y,col+'_smooth':whittaker_smooth(y,w,WH_LAMBDA,WH_ORDER)})

amap=dict(zip(alpha_tbl.district,alpha_tbl.alpha)); tmap=dict(zip(alpha_tbl.district,alpha_tbl.trusted))
def afor(nm):
    a=amap.get(nm,np.nan); return a if (tmap.get(nm,False) and np.isfinite(a)) else alpha_mean
m['alpha']=m.name.map(afor); m['VHI']=m.alpha*m.VCI+(1-m.alpha)*m.TCI
smoothed={nm:smooth_series(g,'VHI') for nm,g in m.groupby('name')}

nm0=sorted(m.name.unique())[0]; s=smoothed[nm0]
plt.figure(figsize=(12,3.5))
plt.plot(s.date,s.VHI,'.',alpha=.4,label='monthly VHI'); plt.plot(s.date,s.VHI_smooth,'-',lw=1.6,label=f'Whittaker λ={WH_LAMBDA}')
plt.axhline(40,color='orange',ls='--',lw=.7); plt.axhline(20,color='red',ls='--',lw=.7)
plt.ylabel('VHI'); plt.title(f'{nm0} — district Landsat VHI (CORINE-211, α calibrated)'); plt.legend()
plt.tight_layout(); plt.show()

## 13 · Export

In [ ]:
allsm=[]
for nm,s in smoothed.items(): s=s.copy(); s['name']=nm; allsm.append(s)
pd.concat(allsm,ignore_index=True).to_csv('trakya_district_vhi_smoothed.csv',index=False)
m.to_csv('trakya_district_vhi_vci_tci_monthly.csv',index=False)
alpha_tbl.to_csv('trakya_district_alpha.csv',index=False)
hand.to_csv('trakya_district_handover.csv',index=False)
match_tbl.to_csv('trakya_gaul_match.csv',index=False)
print('Exported district-level VHI, α table, handover, GAUL match table.')

## Notes
- **If §5 shows CORINE has ~0 pixels over Thrace**, switch to the ESA WorldCover fallback in that
  cell and re-run from §7. Everything downstream is identical.
- **Check §4 match table and §8 NDVI amplitude** first — if amplitude is still low (~0.2), the mask
  or district boundaries are off; if it jumps to ~0.4–0.5, the cropland signal is restored.
- **α now meaningful:** with the CORINE mask, r_VHI should rise well above the v3 ~0. If it does,
  the calibration and the whole VHI–yield validation are back on track.
- **Sensitivity:** try `CROP='Sunflower'` with `GS_MONTHS=[6,7,8]`; report both crops if relevant.
